What we are building (clear goal)

A basic RAG-based Python project that:

Takes a corpus (movies dataset)

Chunks the text

Creates embeddings

Stores them in a vector DB (Chroma)

Accepts a natural language query

Retrieves top-k relevant chunks

Uses an LLM to generate a grounded answer

👉 This is called Naive RAG, and it’s the correct starting point.

In [ ]:
import os
import pandas as pd
import kagglehub

# Download dataset
dataset_path = kagglehub.dataset_download(
    "harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows"
)

# Load CSV
csv_file = os.path.join(dataset_path, "imdb_top_1000.csv")
df = pd.read_csv(csv_file)

In [3]:
import kagglehub

c:\Users\dhanu\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os
import pandas as pd
import kagglehub

# Download dataset
dataset_path = kagglehub.dataset_download(
    "harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows"
)

# Load CSV
csv_file = os.path.join(dataset_path, "imdb_top_1000.csv")
df = pd.read_csv(csv_file)

print(df.columns)
print(df.head())


Index(['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate',
       'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director',
       'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross'],
      dtype='str')
                                         Poster_Link  \
0  https://m.media-amazon.com/images/M/MV5BMDFkYT...   
1  https://m.media-amazon.com/images/M/MV5BM2MyNj...   
2  https://m.media-amazon.com/images/M/MV5BMTMxNT...   
3  https://m.media-amazon.com/images/M/MV5BMWMwMG...   
4  https://m.media-amazon.com/images/M/MV5BMWU4N2...   

               Series_Title Released_Year Certificate  Runtime  \
0  The Shawshank Redemption          1994           A  142 min   
1             The Godfather          1972           A  175 min   
2           The Dark Knight          2008          UA  152 min   
3    The Godfather: Part II          1974           A  202 min   
4              12 Angry Men          1957           U   96 min   

                  Genre  IMDB

In [8]:
print(df.columns)


Index(['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate',
       'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director',
       'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross'],
      dtype='str')


### 2. Chunking movie overviews

In [9]:
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt")

def chunk_text(text, max_length=1500):
    # Safety check
    if not isinstance(text, str) or text.strip() == "":
        return []

    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_length:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dhanu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [11]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")


c:\Users\dhanu\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dhanu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 338.79it/s, Materia

In [15]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dhanu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dhanu\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [28]:
print(df["Overview"].iloc[6])

The lives of two mob hitmen, a boxer, a gangster and his wife, and a pair of diner bandits intertwine in four tales of violence and redemption.


In [ ]:
# Clean Overview first
df = df.dropna(subset=["Overview"])
df["Overview"] = df["Overview"].astype(str)

# Create chunks column
df["chunks"] = df["Overview"].apply(chunk_text) #- apply() takes each element of the "Overview" column (which is a string after your .astype(str)).


In [18]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection(name="movies")

idx = 0
for _, row in df.iterrows():
    title = row["Series_Title"]

    for chunk in row["chunks"]:
        embedding = embedder.encode(chunk).tolist()

        collection.add(
            ids=[str(idx)],
            embeddings=[embedding],
            metadatas=[{
                "title": title,
                "chunk": chunk
            }]
        )
        idx += 1

print(f"Stored {idx} chunks in vector database.")


Stored 1000 chunks in vector database.


#### Retrieval (unchanged)

In [31]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def retrieve_documents(query, collection, top_k=1):
    query_embedding = embedder.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    chunks, titles = [], []
    for meta in results["metadatas"][0]:
        chunks.append(meta["chunk"])
        titles.append(meta["title"])

    return chunks, titles


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 353.73it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [40]:
query = "get me a name which has batman in it"

chunks, titles = retrieve_documents(query, collection)

for t, c in zip(titles, chunks):
    print(f"\n🎬 {t}\n{c}")


#Remember we have encoded Overview column hence I can ask mostly about stories or About of the film



🎬 Batman: Mask of the Phantasm
Batman is wrongly implicated in a series of murders of mob bosses actually done by a new vigilante assassin.
